In [ ]:
conce[t analysid detailed comparisons with original

import pandas as pd
import os
import re
from pathlib import Path
from collections import defaultdict
import json

class LotteryTicketConceptAnalyzer:
    def __init__(self, base_path):
        """
        base_path: Path to model/lottery_ticket directory
        """
        self.base_path = Path(base_path)
        self.prune_data = {}
        
    def load_all_csvs(self):
        """Load all CSV files from the directory structure"""
        print("Loading CSV files...")
        
        # Find all prune level directories
        for prune_dir in sorted(self.base_path.iterdir()):
            if not prune_dir.is_dir():
                continue
                
            # Extract prune level (e.g., "0%prune" -> "0")
            match = re.match(r'(\d+)%', prune_dir.name)
            if not match:
                continue
                
            prune_level = match.group(1)
            self.prune_data[prune_level] = {}
            
            # Load cluster CSVs
            for csv_file in prune_dir.glob(‘Cluster*'):
                cluster_match = re.search(r'cluster(\d+)', csv_file.name)
                if not cluster_match:
                    continue
                    
                cluster = cluster_match.group(1)
                
                try:
                    df = pd.read_csv(csv_file)
                    
                    # Check for required columns
                    if 'unit' not in df.columns or 'formula' not in df.columns:
                        print(f"Warning: {csv_file} missing 'unit' or 'formula' columns")
                        continue
                    
                    # Clean data
                    df = df.dropna(subset=['unit', 'formula'])
                    
                    concepts = set(df['formula'].unique())
                    unit_to_concept = dict(zip(df['unit'], df['formula']))
                    
                    self.prune_data[prune_level][cluster] = {
                        'concepts': concepts,
                        'unit_to_concept': unit_to_concept,
                        'total_units': len(unit_to_concept),
                        'df': df
                    }
                    
                    print(f"  Loaded: {prune_level}% prune, Cluster {cluster} - {len(concepts)} concepts, {len(unit_to_concept)} units")
                    
                except Exception as e:
                    print(f"Error loading {csv_file}: {e}")
        
        print(f"\nLoaded {len(self.prune_data)} pruning levels")
        return self
    
    def get_baseline(self):
        """Get baseline (0% pruning) data"""
        return self.prune_data.get('0', self.prune_data.get(min(self.prune_data.keys())))
    
    def analyze_concept_preservation(self):
        """Analyze how concepts are preserved/lost across pruning levels"""
        baseline = self.get_baseline()
        if not baseline:
            raise ValueError("No baseline data found")
        
        # Get all baseline concepts
        all_baseline_concepts = set()
        for cluster_data in baseline.values():
            all_baseline_concepts.update(cluster_data['concepts'])
        
        print(f"\n{'='*80}")
        print(f"BASELINE: {len(all_baseline_concepts)} unique concepts across all clusters")
        print(f"{'='*80}\n")
        
        results = []
        
        for prune_level in sorted(self.prune_data.keys(), key=int):
            current_data = self.prune_data[prune_level]
            
            print(f"\n{'='*80}")
            print(f"PRUNING LEVEL: {prune_level}%")
            print(f"{'='*80}")
            
            # Per-cluster analysis
            cluster_analysis = {}
            for cluster in sorted(current_data.keys(), key=int):
                baseline_cluster = baseline.get(cluster)
                current_cluster = current_data[cluster]
                
                if not baseline_cluster:
                    continue
                
                baseline_concepts = baseline_cluster['concepts']
                current_concepts = current_cluster['concepts']
                
                preserved = current_concepts & baseline_concepts
                lost = baseline_concepts - current_concepts
                new_concepts = current_concepts - baseline_concepts
                
                preservation_rate = (len(preserved) / len(baseline_concepts) * 100) if baseline_concepts else 0
                
                cluster_analysis[cluster] = {
                    'preserved': len(preserved),
                    'lost': len(lost),
                    'new': len(new_concepts),
                    'preserved_list': sorted(preserved),
                    'lost_list': sorted(lost),
                    'new_list': sorted(new_concepts),
                    'total_units': current_cluster['total_units'],
                    'baseline_units': baseline_cluster['total_units'],
                    'preservation_rate': preservation_rate
                }
                
                print(f"\n  Cluster {cluster}:")
                print(f"    Units: {current_cluster['total_units']} / {baseline_cluster['total_units']} ({current_cluster['total_units']/baseline_cluster['total_units']*100:.1f}%)")
                print(f"    Concepts Preserved: {len(preserved)} ({preservation_rate:.1f}%)")
                print(f"    Concepts Lost: {len(lost)}")
                if new_concepts:
                    print(f"    New Concepts: {len(new_concepts)}")
                
                if lost and len(lost) <= 10:
                    print(f"    Lost: {', '.join(sorted(lost))}")
                elif lost:
                    print(f"    Lost (sample): {', '.join(list(sorted(lost))[:5])}...")
            
            # Global analysis
            all_current_concepts = set()
            for cluster_data in current_data.values():
                all_current_concepts.update(cluster_data['concepts'])
            
            global_preserved = all_current_concepts & all_baseline_concepts
            global_lost = all_baseline_concepts - all_current_concepts
            global_new = all_current_concepts - all_baseline_concepts
            
            global_preservation_rate = (len(global_preserved) / len(all_baseline_concepts) * 100)
            
            print(f"\n  GLOBAL SUMMARY:")
            print(f"    Total Concepts: {len(all_current_concepts)} / {len(all_baseline_concepts)}")
            print(f"    Preservation Rate: {global_preservation_rate:.1f}%")
            print(f"    Concepts Lost Globally: {len(global_lost)}")
            if global_new:
                print(f"    New Concepts: {len(global_new)}")
            
            if global_lost:
                print(f"\n    Globally Lost Concepts:")
                for concept in sorted(global_lost):
                    print(f"      - {concept}")
            
            results.append({
                'prune_level': prune_level,
                'clusters': cluster_analysis,
                'global': {
                    'preserved': len(global_preserved),
                    'lost': len(global_lost),
                    'new': len(global_new),
                    'preservation_rate': global_preservation_rate,
                    'lost_concepts': sorted(global_lost),
                    'new_concepts': sorted(global_new)
                }
            })
        
        return {
            'analysis': results,
            'baseline_concept_count': len(all_baseline_concepts),
            'baseline_concepts': sorted(all_baseline_concepts)
        }
    
    def concept_trajectory(self, concept_name):
        """Track a specific concept across pruning levels"""
        print(f"\n{'='*80}")
        print(f"CONCEPT TRAJECTORY: '{concept_name}'")
        print(f"{'='*80}\n")
        
        for prune_level in sorted(self.prune_data.keys(), key=int):
            print(f"\nPruning {prune_level}%:")
            current_data = self.prune_data[prune_level]
            
            found_in_clusters = []
            for cluster, cluster_data in current_data.items():
                if concept_name in cluster_data['concepts']:
                    # Count units with this concept
                    units = [u for u, c in cluster_data['unit_to_concept'].items() if c == concept_name]
                    found_in_clusters.append((cluster, len(units)))
            
            if found_in_clusters:
                for cluster, count in found_in_clusters:
                    print(f"  Cluster {cluster}: {count} units")
            else:
                print(f"  *** CONCEPT LOST ***")
    
    def cluster_concept_overlap(self):
        """Analyze how concepts are distributed across clusters"""
        print(f"\n{'='*80}")
        print(f"CLUSTER CONCEPT OVERLAP ANALYSIS")
        print(f"{'='*80}\n")
        
        for prune_level in sorted(self.prune_data.keys(), key=int):
            print(f"\nPruning {prune_level}%:")
            current_data = self.prune_data[prune_level]
            
            # Count concepts per cluster
            concept_to_clusters = defaultdict(set)
            for cluster, cluster_data in current_data.items():
                for concept in cluster_data['concepts']:
                    concept_to_clusters[concept].add(cluster)
            
            # Categorize
            unique_concepts = {c: clusters for c, clusters in concept_to_clusters.items() if len(clusters) == 1}
            shared_concepts = {c: clusters for c, clusters in concept_to_clusters.items() if len(clusters) > 1}
            
            print(f"  Unique to one cluster: {len(unique_concepts)}")
            print(f"  Shared across clusters: {len(shared_concepts)}")
            
            if shared_concepts and len(shared_concepts) <= 20:
                print(f"\n  Shared concepts:")
                for concept, clusters in sorted(shared_concepts.items()):
                    print(f"    {concept}: clusters {sorted(clusters)}")
    
    def save_results(self, results, output_file='concept_analysis_results.json'):
        """Save analysis results to JSON"""
        # Convert sets to lists for JSON serialization
        def convert_sets(obj):
            if isinstance(obj, set):
                return sorted(list(obj))
            elif isinstance(obj, dict):
                return {k: convert_sets(v) for k, v in obj.items()}
            elif isinstance(obj, list):
                return [convert_sets(item) for item in obj]
            return obj
        
        results_serializable = convert_sets(results)
        
        with open(output_file, 'w') as f:
            json.dump(results_serializable, f, indent=2)
        print(f"\n\nResults saved to {output_file}")


# Example usage
if __name__ == "__main__":
    # Set your path here
    base_path = "model/lottery_ticket"  # Adjust this path
    
    analyzer = LotteryTicketConceptAnalyzer(base_path)
    analyzer.load_all_csvs()
    
    # Run main analysis
    results = analyzer.analyze_concept_preservation()
    
    # Save results
    analyzer.save_results(results)
    
    # Track specific concept (example)
    # Uncomment to track a specific concept:
    # analyzer.concept_trajectory("the dog")
    
    # Analyze cluster overlap
    analyzer.cluster_concept_overlap()
    
    print("\n\nAnalysis complete!")


"""
EXAMPLE OUTPUT:
================================================================================

Loading CSV files...
  Loaded: 0% prune, Cluster 1 - 95 concepts, 612 units
  Loaded: 0% prune, Cluster 2 - 142 concepts, 1843 units
  Loaded: 0% prune, Cluster 3 - 178 concepts, 2156 units

Loaded 6 pruning levels

================================================================================
BASELINE: 415 unique concepts across all clusters
================================================================================


================================================================================
PRUNING LEVEL: 0%
================================================================================

  Cluster 1:
    Units: 612 / 612 (100.0%)
    Concepts Preserved: 95 (100.0%)
    Concepts Lost: 0

  Cluster 2:
    Units: 1843 / 1843 (100.0%)
    Concepts Preserved: 142 (100.0%)
    Concepts Lost: 0

  Cluster 3:
    Units: 2156 / 2156 (100.0%)
    Concepts Preserved: 178 (100.0%)
    Concepts Lost: 0

  GLOBAL SUMMARY:
    Total Concepts: 415 / 415
    Preservation Rate: 100.0%
    Concepts Lost Globally: 0


================================================================================
PRUNING LEVEL: 25%
================================================================================

  Cluster 1:
    Units: 459 / 612 (75.0%)
    Concepts Preserved: 88 (92.6%)
    Concepts Lost: 7
    Lost: [in], [on], [the cat], [with], a dog, red car, the house

  Cluster 2:
    Units: 1382 / 1843 (75.0%)
    Concepts Preserved: 135 (95.1%)
    Concepts Lost: 7
    New Concepts: 2
    Lost: [after], blue sky, green tree, small box, under table, very cold, yellow bird

  Cluster 3:
    Units: 1617 / 2156 (75.0%)
    Concepts Preserved: 171 (96.1%)
    Concepts Lost: 7
    New Concepts: 3
    Lost: big house, fast car, hot coffee, old man, tall building, warm day, young child

  GLOBAL SUMMARY:
    Total Concepts: 399 / 415
    Preservation Rate: 96.1%
    Concepts Lost Globally: 16
    New Concepts: 5

    Globally Lost Concepts:
      - [in]
      - [on]
      - [the cat]
      - [with]
      - a dog
      - big house
      - fast car
      - hot coffee
      - old man
      - red car
      - tall building
      - the house
      - under table
      - very cold
      - warm day
      - young child


================================================================================
PRUNING LEVEL: 43%
================================================================================

  Cluster 1:
    Units: 349 / 612 (57.0%)
    Concepts Preserved: 76 (80.0%)
    Concepts Lost: 19
    Lost (sample): [a], [and], [at], [for], [from]...

  Cluster 2:
    Units: 1050 / 1843 (57.0%)
    Concepts Preserved: 118 (83.1%)
    Concepts Lost: 24
    New Concepts: 4

  Cluster 3:
    Units: 1229 / 2156 (57.0%)
    Concepts Preserved: 154 (86.5%)
    Concepts Lost: 24
    New Concepts: 8

  GLOBAL SUMMARY:
    Total Concepts: 360 / 415
    Preservation Rate: 86.7%
    Concepts Lost Globally: 55
    New Concepts: 12

    Globally Lost Concepts:
      - [a]
      - [and]
      - [at]
      - [for]
      - [from]
      - [in]
      - [of]
      - [on]
      - [the]
      - [the cat]
      - [to]
      - [with]
      - a dog
      - beautiful flower
      - big house
      - black cat
      ... (39 more)


================================================================================
PRUNING LEVEL: 76%
================================================================================

  Cluster 1:
    Units: 147 / 612 (24.0%)
    Concepts Preserved: 42 (44.2%)
    Concepts Lost: 53
    New Concepts: 1

  Cluster 2:
    Units: 442 / 1843 (24.0%)
    Concepts Preserved: 81 (57.0%)
    Concepts Lost: 61
    New Concepts: 7

  Cluster 3:
    Units: 518 / 2156 (24.0%)
    Concepts Preserved: 98 (55.1%)
    Concepts Lost: 80
    New Concepts: 15

  GLOBAL SUMMARY:
    Total Concepts: 244 / 415
    Preservation Rate: 58.8%
    Concepts Lost Globally: 171
    New Concepts: 23

    Globally Lost Concepts:
      - [a]
      - [after]
      - [all]
      - [an]
      - [and]
      - [around]
      - [as]
      - [at]
      - [before]
      - [between]
      - [but]
      - [by]
      - [for]
      - [from]
      - [in]
      ... (156 more - mostly function words and low-level concepts)


================================================================================
CONCEPT TRAJECTORY: 'the dog'
================================================================================

Pruning 0%:
  Cluster 2: 12 units
  Cluster 3: 8 units

Pruning 25%:
  Cluster 2: 9 units
  Cluster 3: 6 units

Pruning 43%:
  Cluster 2: 5 units
  Cluster 3: 3 units

Pruning 57%:
  Cluster 3: 2 units

Pruning 68%:
  Cluster 3: 1 units

Pruning 76%:
  *** CONCEPT LOST ***


================================================================================
CLUSTER CONCEPT OVERLAP ANALYSIS
================================================================================

Pruning 0%:
  Unique to one cluster: 326
  Shared across clusters: 89

  Shared concepts (sample):
    [the]: clusters ['1', '2', '3']
    beautiful: clusters ['2', '3']
    person: clusters ['2', '3']
    running: clusters ['1', '3']
    the dog: clusters ['2', '3']
    walking: clusters ['2', '3']

Pruning 25%:
  Unique to one cluster: 318
  Shared across clusters: 81

Pruning 43%:
  Unique to one cluster: 295
  Shared across clusters: 65

Pruning 76%:
  Unique to one cluster: 198
  Shared across clusters: 46
  
  Shared concepts:
    animal: clusters ['2', '3']
    eat: clusters ['2', '3']
    house: clusters ['1', '3']
    person: clusters ['2', '3']
    sleep: clusters ['2', '3']
    water: clusters ['2', '3']


Results saved to concept_analysis_results.json

Analysis complete!
"""